<a href="https://colab.research.google.com/github/hadalulu/legendary-journey/blob/main/scripts/Hada_Lul%C3%BA_Narration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import userdata
api_key = userdata.get('XAI_API_KEY')

In [ ]:
#!/usr/bin/env python3
"""Generate one Spanish WAV narration file for every storybook page.

Uses xAI's Text-to-Speech REST API. Set XAI_API_KEY in your environment
before running this script. Generated speech segments are cached locally so
an interrupted run can resume without paying to synthesize completed lines.
"""

from __future__ import annotations

import hashlib
import json
import os
import io
from pathlib import Path
import random
import struct
import math
import sys
import time
import urllib.error
import urllib.request
import wave


API_URL = "https://api.x.ai/v1/tts"
LANGUAGE = "es-MX"
SAMPLE_RATE = 44_100
PAUSE_MS = 360
EFFECT_VERSION = 3
EFFECT_FILES = {
    "loud_fart": Path("loud-fart-public-domain.wav")
}

# The four goblins share a base voice so they sound related, but use distinct
# rates and documented xAI speech tags for clearly different personalities.
VOICES = {
    "narrator": {"voice": "ara", "speed": 0.88},
    "lulu": {"voice": "eve", "speed": 0.98},
    "emma": {"voice": "aurora", "speed": 0.95},
    "raquel": {"voice": "celeste", "speed": 0.96},
    "goblin_1": {"voice": "cosmo", "speed": 1.05},
    "goblin_2": {"voice": "cosmo", "speed": 1.12},
    "goblin_3": {"voice": "cosmo", "speed": 1.00},
    "goblin_4": {"voice": "cosmo", "speed": 1.16},
    "tiger": {"voice": "zagan", "speed": 0.88},
}

def line(role: str, text: str) -> dict[str, str]:
    return {"role": role, "text": text}

def effect(name: str) -> dict[str, str]:
    return {"kind": "effect", "name": name}


In [51]:

# Speech tags are sent to xAI, but are not spoken aloud. Each inner list is one
# finished page; its segments are synthesized separately, then joined to WAV.
PAGES = [
    [line("narrator", "<soft><slow>La Poderosa Hada Lulú.</slow></soft>")],
    [line("narrator", "Había una vez un hada muy hermosa y muy poderosa llamada Lulú. [pause]")],
    [line("narrator", "Las hadas son muy pequeñitas, más o menos del tamaño de un dedo. Tienen alas brillantes como las de una mariposa y viven en un bosque encantado, muy, muy lejos de las casas de los niños. Las hadas tienen muchos poderes mágicos: pueden volar muy rápido, hablar con los animalitos, hacerse invisibles y, con un movimiento de su varita, hacer que las flores y las plantas crezcan al instante.")],
    [line("narrator", "El hada Lulú era una de las hadas más poderosas y volaba más rápido que todas las demás. [pause] Además, tenía un pequeño tigre muy especial. Cuando Lulú le hacía un guiño, el tigrito crecía de repente y se convertía en un enorme tigre protector. Con un gran rugido espantaba a los malos, y todos salían corriendo.")],
    [line("narrator", "¿Sabes quiénes son los malos que siempre enojan a las hadas, Lulu? [pause] Son los duendes. [pause] Los duendes son unas criaturitas pequeñitas, del tamaño de un peluche. Tienen la piel verde, orejas grandes y puntiagudas, una nariz redonda y una sonrisa <emphasis>traviesa</emphasis>. Les encanta hacer <emphasis>bromas</emphasis>, como esconder un calcetín, mover un juguete de lugar o hacer <emphasis>mucho</emphasis> desorden. No son malos de verdad. Solo son <emphasis>muy, muy traviesos</emphasis> y casi <emphasis>nunca</emphasis> piensan antes de hacer una <emphasis>travesura.</emphasis>")],
    #[line("narrator", "Cada noche, las hadas salen volando de su bosque encantado para visitar, en secreto, las casas de los niños mientras duermen. Llevan consigo un polvito <emphasis>mágico</emphasis> y <emphasis>brillante</emphasis> llamado <emphasis>polvo de hada</emphasis>, que ayuda a que los niños tengan sueños <emphasis>felices y tranquilos</emphasis>. [pause] Entran muy despacito en cada habitación y dejan caer un poquito de polvo sobre los niños dormidos. Entonces, los niños sueñan con aventuras, arcoíris, dinosaurios amistosos y cachorritos juguetones.")],
    [
    line(
        "narrator",
        "Cada noche, las hadas salen volando de su bosque encantado para visitar, "
        "en secreto, las casas de los niños mientras duermen. Llevan consigo un "
        "polvito <emphasis>mágico</emphasis> y <emphasis>brillante</emphasis> "
        "llamado <emphasis>polvo de hada</emphasis>, que ayuda a que los niños "
        "tengan sueños <emphasis>felices y tranquilos</emphasis>. [pause] "
        "Entran <soft><slow>muy despacito</slow></soft> en cada habitación y "
        "dejan caer un poquito de polvo sobre los niños dormidos. Entonces, los "
        "niños sueñan con <soft><sing-song>aventuras... [pause] arcoíris... "
        "[pause] dinosaurios amistosos... y cachorritos juguetones."
        "</sing-song></soft>"
        )
    ],
    [line("narrator", "Una noche, justo cuando el hada Lulú iba a salir del bosque encantado, escuchó unas risitas detrás de un árbol."), line("goblin_2", "<whisper>[giggle] Je, je, je...</whisper>"), line("narrator", "Lulú se acercó muy despacito para escuchar.")],
    [
    line(
        "narrator",
        "[inhale] <loud><emphasis>¡Eran cuatro duendes!</emphasis></loud>"
    ),
    line(
        "goblin_1",
        "<higher-pitch>[giggle] "
        "<sing-song>¡Hoy vamos a hacer la travesura más grande de todas!</sing-song>"
        "</higher-pitch>"
    ),
    line("narrator", "dijo uno riéndose."),
    line(
        "goblin_2",
        "<higher-pitch>[inhale] "
        "<fast>¡Vamos a llevarnos todo el polvo de hada!</fast> "
        "[giggle]</higher-pitch>"
    ),
    line("narrator", "dijo otro."),
    line(
        "goblin_3",
        "<higher-pitch><sing-song>¡Después lo llevaremos a los otros duendes que nos están esperando en el bosque, y lo vamos a aventar por los aires para hacer una nube brillante!</sing-song> [giggle] ¡Je, je, je!</higher-pitch>"),
    line(
        "goblin_4",
        "<higher-pitch>[tongue-click] [giggle] "
        "<sing-song>¡Y yo me voy a lavar las pompis con el polvo de hada!</sing-song>"
        "</higher-pitch>"
    ),
    line(
        "narrator",
        "dijo un duende muy orgulloso. Los otros empezaron a reírse."
    ),
    effect("loud_fart"),
    line(
        "goblin_1",
        "<higher-pitch>[tsk] <emphasis>¡Puaj!</emphasis> "
        "¡Te echaste un pedito! [giggle]</higher-pitch>"
    ),
    line(
        "goblin_2",
        "<higher-pitch>[giggle] ¡Je, je, je! "
        "<fast>¡Otro, otro!</fast></higher-pitch>"
    ),
    effect("loud_fart"),
    line(
        "goblin_4",
        "<higher-pitch><sing-song>"
        "¡El próximo pedito será con polvo mágico!"
        "</sing-song> [giggle] ¡Je, je, je!</higher-pitch>"
    ),
    line("narrator", "Los cuatro duendes se morían de la risa."),
    line("goblin_1", "<higher-pitch>[chuckle]</higher-pitch>"),
    line("goblin_2", "<higher-pitch>[giggle]</higher-pitch>"),
    line("goblin_3", "<higher-pitch>[laugh]</higher-pitch>"),
    line("goblin_4", "<higher-pitch>[giggle] [laugh]</higher-pitch>")
],
       #[line("narrator", "[inhale] <loud><emphasis>¡Eran cuatro duendes!</emphasis></loud>"), line("goblin_1", "<higher-pitch>[chuckle] <build-intensity>¡Hoy vamos a hacer la travesura más grande de todas!</build-intensity></higher-pitch>"), line("narrator", "dijo uno riéndose."), line("goblin_2", "<higher-pitch>[giggle] <fast>¡Vamos a llevarnos todo el polvo de hada!</fast></higher-pitch>"), line("narrator", "dijo otro."), line("goblin_3", "<higher-pitch><sing-song>¡Después lo llevaremos a los otros duendes que nos están esperando en el bosque, y lo vamos a aventar por los aires para hacer una nube brillante!</sing-song> [giggle] ¡Je, je, je!</higher-pitch>"), line("goblin_4", "<higher-pitch>[tongue-click] <sing-song>¡Y yo me voy a lavar las pompis con el polvo de hada!</sing-song></higher-pitch>"), line("narrator", "dijo un duende muy orgulloso. Los otros empezaron a reírse."), effect("loud_fart"), line("goblin_1", "<higher-pitch>[tsk] ¡Puaj! ¡Te echaste un pedito!</higher-pitch>"), line("goblin_2", "<higher-pitch>[giggle] ¡Je, je, je! <fast>¡Otro, otro!</fast></higher-pitch>"), effect("loud_fart"), line("goblin_4", "<higher-pitch><sing-song>¡El próximo pedito será con polvo mágico!</sing-song> [giggle] ¡Je, je, je!</higher-pitch>"), line("narrator", "Los cuatro duendes se morían de la risa."), line("goblin_1", "<higher-pitch>[chuckle]</higher-pitch>"), line("goblin_2", "<higher-pitch>[giggle]</higher-pitch>"), line("goblin_3", "<higher-pitch>[laugh]</higher-pitch>"), line("goblin_4", "<higher-pitch>[giggle] [laugh]</higher-pitch>")],
    [line("narrator", "Lulú abrió mucho los ojos."), line("lulu", "<build-intensity><loud>¡Oh, no!</loud>[pause] Si se llevan todo el polvo de hada, las hadas ya no podrán llevar sueños felices a los niños en las próximas noches.</build-intensity> [pause] <emphasis>¡Ya sé!</emphasis> Yo soy el hada <emphasis>más rápida</emphasis>. Iré por ayuda."), line("narrator", "Y salió volando <emphasis>tan, tan</emphasis> rápido que parecía una estrella cruzando el cielo.")],
    [line("narrator", "Muy pronto encontró a sus amigas, las hadas Emma y Raquel."), line("lulu", "¡Los duendes quieren llevarse el polvo mágico!"), line("emma", "<loud>¡Vamos!</loud>"), line("raquel", "<loud>¡Vamos!</loud>"), line("lulu", "<loud>¡Vamos!</loud>"), line("narrator", "Las tres hadas llegaron al castillo tan rápido como pudieron y corrieron hasta la sala donde guardaban el polvo de hada.")],
    [line("lulu", "[inhale] ¡Ay, no!"), line("narrator", "El gran cofre del polvo de hada estaba abierto. [long-pause] ¡Y estaba completamente vacío! [long-pause] Por un momento, las tres hadas se quedaron en silencio.")],
    [line("narrator", "Emma señaló por la ventana."), line("emma", "<build-intensity>¡Miren! ¡Sale humo del bosque! Si no lo apagamos, podría convertirse en un gran incendio. ¡Yo iré!</build-intensity>"), line("narrator", "Raquel miró hacia una colina cercana."), line("raquel", "¡Y allá hay un grupo enorme de duendes esperando a los cuatro ladrones! ¡Yo voy a detenerlos!"), line("narrator", "Lulú vio un caminito de polvo brillante que se perdía entre los árboles."), line("lulu", "Yo seguiré el rastro, recuperaré el polvo de hada y luego nos reuniremos."), line("narrator", "Y las tres salieron volando, cada una hacia su importante misión.")],
    [line("narrator", "Emma siguió el humo hasta encontrar una fogata que los duendes habían dejado encendida. Levantó su varita."), line("emma", "<loud>¡Lluvia mágica!</loud>"), line("narrator", "Al instante comenzó a llover. ¡Plin, plin, plin! En pocos segundos, el fuego se apagó."), line("emma", "[sigh] ¡El bosque está a salvo!")],
    [line("narrator", "Mientras tanto, Raquel encontró al gran grupo de duendes esperando a los ladrones. Levantó su varita."), line("raquel", "<loud>¡Enredaderas, crezcan ahora!</loud>"), line("narrator", "Al instante, unas enredaderas crecieron del suelo y atraparon a todos los duendes."), line("raquel", "¡Listo! ¡Ahora los ladrones no tendrán dónde esconderse!")],
    [line("narrator", "Al mismo tiempo, Lulú siguió el caminito de polvo brillante y encontró a los cuatro duendes que llevaban el saco de polvo de hada."), line("goblin_1", "<lower-pitch><loud>¡Ni se te ocurra usar tu magia!</loud> Si levantas tu varita, romperemos el saco y todo el polvo se perderá.</lower-pitch>"), line("narrator", "Lulú sonrió."), line("lulu", "Muy bien. Entonces... no usaré mi magia."), line("narrator", "Los duendes empezaron a reír."), line("goblin_2", "<higher-pitch>[giggle] Je, je, je.</higher-pitch>"), line("goblin_3", "[chuckle]"), line("goblin_4", "[laugh]")],
    [line("narrator", "Pero Lulú le guiñó un ojo a su tigrito. El tigrito dio un saltito. [pause] ¡Puf! En un abrir y cerrar de ojos se convirtió en un enorme tigre protector."), line("tiger", "<lower-pitch><loud>¡Rooooaaar!</loud></lower-pitch>"), line("narrator", "Los duendes dieron un brinco del susto y salieron corriendo, dejando el saco de polvo de hada en el suelo. Lulú lo recogió y sonrió."), line("lulu", "¡Lo recuperamos!")],
    [line("narrator", "Lulú voló hasta la colina donde estaban Emma y Raquel. Allí, Raquel había atrapado al gran grupo de duendes. Poco después llegaron corriendo los cuatro ladrones y las enredaderas también los atraparon."), line("lulu", "¡Lo logramos!"), line("emma", "¡Y el bosque está a salvo!"), line("narrator", "Pero Lulú se quedó pensativa."), line("lulu", "Ahora que conocen el camino al castillo... ¿qué podemos hacer para que no vuelvan?")],
    [line("lulu", "Necesitamos hacer un hechizo muy poderoso. Pero solo funciona si las tres hadas... ¡y la niña Lulú!... respiran juntas. [pause] ¿Nos ayudas, niña Lulú?"), line("narrator", "Sí, ¡a ti! Estamos hablando contigo, nuestra pequeña lectora. Sin tu ayuda, el hechizo no funcionará.")],
    [line("narrator", "<soft><slow>Primero imaginemos que olemos una flor muy bonita. [pause] Respira hondo... [inhale] Uno... dos... tres... [pause] Ahora sopla una velita. [exhale] Fuuuu... [long-pause] Otra vez. [inhale] Uno... dos... tres... [exhale] Fuuuu... [long-pause] Y una última vez. [inhale] Uno... dos... tres... [exhale] Fuuuu...</slow></soft>")],
    [line("narrator", "Las tres hadas apuntaron sus varitas al cielo."), line("lulu", "<build-intensity>¡Luz de luna,</build-intensity>"), line("emma", "<build-intensity>brillo de estrella,</build-intensity>"), line("raquel", "<build-intensity>que los duendes olviden el camino al castillo y recuerden siempre el camino a su hogar!</build-intensity>"), line("narrator", "[long-pause] <loud>¡Zas!</loud>")],
    [line("narrator", "Los duendes parpadearon."), line("goblin_1", "<higher-pitch>¿Qué estábamos haciendo?</higher-pitch>"), line("goblin_2", "<higher-pitch>¡No me acuerdo!</higher-pitch>"), line("goblin_4", "<higher-pitch><sing-song>¡Vamos a casa!</sing-song> [giggle]</higher-pitch>"), line("narrator", "Y se fueron riendo por el bosque."), line("goblin_3", "[chuckle]")],
    [line("narrator", "Lulú guardó el polvo de hada en su lugar. Después, las tres hadas salieron volando para llevar sueños felices a los niños.")],
    [line("narrator", "Lulú miró hacia el cielo y sonrió."), line("lulu", "<soft>Misión cumplida.</soft>"), line("narrator", "<sing-song>Y colorín colorado... ¡este cuento se ha acabado!</sing-song>")],
]


def request_wav(api_key: str, role: str, text: str) -> bytes:
    profile = VOICES[role]
    payload = json.dumps(
        {
            "text": text,
            "voice_id": profile["voice"],
            "language": LANGUAGE,
            "speed": profile["speed"],
            "text_normalization": True,
            "output_format": {"codec": "wav", "sample_rate": SAMPLE_RATE},
        }
    ).encode("utf-8")
    request = urllib.request.Request(
        API_URL,
        data=payload,
        method="POST",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "Accept": "audio/wav",
        },
    )
    for attempt in range(4):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                audio = response.read()
            if not audio.startswith(b"RIFF"):
                raise RuntimeError("xAI returned data that is not a WAV file")
            return audio
        except urllib.error.HTTPError as exc:
            details = exc.read().decode("utf-8", errors="replace")
            if exc.code not in {429, 500, 502, 503, 504} or attempt == 3:
                raise RuntimeError(f"xAI TTS failed ({exc.code}): {details}") from exc
        except urllib.error.URLError as exc:
            if attempt == 3:
                raise RuntimeError(f"Could not reach xAI TTS: {exc.reason}") from exc
        time.sleep(2**attempt)
    raise RuntimeError("xAI TTS request failed")


def cached_effect(cache_dir: Path, name: str, force: bool) -> Path:
    path = cache_dir / f"effect-{name}-v{EFFECT_VERSION}.wav"
    if force or not path.exists():
        source = EFFECT_FILES.get(name)
        if source is None:
            raise ValueError(f"Unknown sound effect: {name}")
        if not source.exists():
            raise FileNotFoundError(f"Bundled sound effect is missing: {source}")
        path.write_bytes(source.read_bytes())
    return path



In [49]:
def cached_segment(
    api_key: str, cache_dir: Path, role: str, text: str, force: bool
) -> Path:
    profile = VOICES[role]
    cache_key = json.dumps(
        {"role": role, "text": text, **profile, "language": LANGUAGE},
        ensure_ascii=False,
        sort_keys=True,
    )
    digest = hashlib.sha256(cache_key.encode("utf-8")).hexdigest()[:20]
    path = cache_dir / f"{digest}.wav"
    if force or not path.exists():
        path.write_bytes(request_wav(api_key, role, text))
    return path


def read_wav(path: Path) -> tuple[wave._wave_params, bytes]:
    with wave.open(str(path), "rb") as source:
        return source.getparams(), source.readframes(source.getnframes())


def join_wavs(segments_data: list[tuple[Path, int]], output_path: Path) -> None:
    if not segments_data:
        raise ValueError("No audio segments to join.")

    all_frames_to_write = []

    # Read the first segment's path and establish the output WAV parameters
    first_path, _ = segments_data[0]
    first_params, _ = read_wav(first_path) # Just to get parameters for the target WAV file

    for segment_path, pause_ms_after in segments_data:
        # Read the current audio segment's frames
        current_params, current_frames = read_wav(segment_path)

        # Validate that current segment's format is compatible with the first segment's format
        if (current_params.nchannels, current_params.sampwidth, current_params.framerate, current_params.comptype) != \
           (first_params.nchannels, first_params.sampwidth, first_params.framerate, first_params.comptype):
            raise RuntimeError(f"Incompatible WAV format in {segment_path}")

        all_frames_to_write.append(current_frames)

        # Append silence after the current segment based on its 'pause_ms_after' value
        if pause_ms_after > 0:
            silence_frames = round(first_params.framerate * pause_ms_after / 1000)
            # Create silence chunk based on the determined WAV parameters
            silence_chunk = b"\x00" * silence_frames * first_params.nchannels * first_params.sampwidth
            all_frames_to_write.append(silence_chunk)

    # Write all collected audio frames and silences to the output WAV file
    with wave.open(str(output_path), "wb") as target:
        target.setnchannels(first_params.nchannels)
        target.setsampwidth(first_params.sampwidth)
        target.setframerate(first_params.framerate)
        target.setcomptype(first_params.comptype, first_params.compname)
        target.writeframes(b"".join(all_frames_to_write))


def generate_pages(api_key: str, output_dir="/content/audio/narration", pages=None, force=False):
    """Generate selected 1-based pages (or all pages) in Google Colab."""
    output_dir = Path(output_dir).resolve()
    cache_dir = output_dir / ".segments"
    cache_dir.mkdir(parents=True, exist_ok=True)

    if pages is None:
        page_numbers = list(range(1, len(PAGES) + 1))
    elif isinstance(pages, int):
        page_numbers = [pages]
    else:
        page_numbers = sorted(set(int(p) for p in pages))

    invalid = [p for p in page_numbers if not 1 <= p <= len(PAGES)]
    if invalid:
        raise ValueError(f"Invalid page number(s): {invalid}; valid range is 1-{len(PAGES)}")

    generated = []
    for page_number in page_numbers:
        destination = output_dir / f"page-{page_number:02d}.wav"
        print(f"Page {page_number:02d}/{len(PAGES)} -> {destination}")
        segments = []
        for item in PAGES[page_number - 1]:
            if item.get("kind") == "effect":
                print(f"  effect: {item['name']}")
                segments.append(
                    (cached_effect(cache_dir, item["name"], force), 0)
                )
                continue
            print(f"  {item['role']}: {item['text'][:54]}...")
            segments.append(
                (
                    cached_segment(
                        api_key, cache_dir, item["role"], item["text"], force
                    ),
                    PAUSE_MS,
                )
            )
        join_wavs(segments, destination)
        generated.append(destination)

    print(f"Done. Generated {len(generated)} page file(s) in {output_dir}")
    return generated

In [53]:
generate_pages(api_key, output_dir="/content/audio/narration", pages=10, force=False)

Page 10/23 -> /content/audio/narration/page-10.wav
  narrator: Muy pronto encontró a sus amigas, las hadas Emma y Raq...
  lulu: ¡Los duendes quieren llevarse el polvo mágico!...
  emma: <loud>¡Vamos!</loud>...
  raquel: <loud>¡Vamos!</loud>...
  lulu: <loud>¡Vamos!</loud>...
  narrator: Las tres hadas llegaron al castillo tan rápido como pu...
Done. Generated 1 page file(s) in /content/audio/narration


[PosixPath('/content/audio/narration/page-10.wav')]

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')